In [194]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from sklearn.preprocessing import OrdinalEncoder

print ('Setup completed')

Setup completed


In [195]:
# Загрузка данных
full_data_test = pd.read_csv('../data/test.csv')
full_data_train = pd.read_csv('../data/train.csv')

y = full_data_train.SalePrice
X = full_data_train.drop('SalePrice', axis=1)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size = 0.2, random_state = 42)


In [196]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

Работа с числовыми колонками

In [197]:
# Работа с Nan
nan_indx = X_train.index[X_train['MasVnrArea'].isna()]
X_train.drop(nan_indx, inplace=True)
y_train.drop(nan_indx, inplace=True)

X_train['Garage_exists'] = X_train['GarageYrBlt'].notna().astype(int)
X_train.drop('GarageYrBlt', axis=1, inplace=True)
X_valid['Garage_exists'] = X_valid['GarageYrBlt'].notna().astype(int)
X_valid.drop('GarageYrBlt', axis=1, inplace=True)

num_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])

In [198]:
# Проверка гипотез числовых данных
# Возраст дома на момент продажи (RForest +11, XGboost -100)
X_train['HouseAge'] = X_train['YrSold'] - X_train['YearBuilt']
X_valid['HouseAge'] = X_valid['YrSold'] - X_valid['YearBuilt']

# Возраст после ремонта (Rforest -25(17820), XGboost -110(16221)
X_train['RemodAge'] = X_train['YrSold'] - X_train['YearRemodAdd']
X_valid['RemodAge'] = X_valid['YrSold'] - X_valid['YearRemodAdd']

#Был ли ремонт (RForest -14(17806), XGboost -116 (16105))
X_train['WasRemodeled'] = (X_train['YearBuilt'] != X_train['YearRemodAdd']).astype(int)
X_valid['WasRemodeled'] = (X_valid['YearBuilt'] != X_valid['YearRemodAdd']).astype(int)

# Общая площадь (RForest -236 (17570), XGboost +165 (16270))
X_train['TotalSF'] = X_train['TotalBsmtSF'] + X_train['1stFlrSF'] + X_train['2ndFlrSF']
X_valid['TotalSF'] = X_valid['TotalBsmtSF'] + X_valid['1stFlrSF'] + X_valid['2ndFlrSF']

# Общая площадь крыльца/террас (RForest +21 (17591) , -192 (16078))
X_train['TotalPorchSF'] = (
    X_train['OpenPorchSF'] +
    X_train['EnclosedPorch'] +
    X_train['3SsnPorch'] +
    X_train['ScreenPorch'] +
    X_train['WoodDeckSF']
)
X_valid['TotalPorchSF'] = (
    X_valid['OpenPorchSF'] +
    X_valid['EnclosedPorch'] +
    X_valid['3SsnPorch'] +
    X_valid['ScreenPorch'] +
    X_valid['WoodDeckSF']
)

# Общее число ванных (RForest -164(17426), XGboost -11 (16069))
X_train['TotalBath'] = (
    X_train['FullBath'] +
    0.5 * X_train['HalfBath'] +
    X_train['BsmtFullBath'] +
    0.5 * X_train['BsmtHalfBath']
)
X_valid['TotalBath'] = (
    X_valid['FullBath'] +
    0.5 * X_valid['HalfBath'] +
    X_valid['BsmtFullBath'] +
    0.5 * X_valid['BsmtHalfBath']
)

# Есть ли бассейн (RForest -65(17361), XGboost -46 (16023))
X_train['HasPool'] = (X_train['PoolArea'] > 0).astype(int)
X_valid['HasPool'] = (X_valid['PoolArea'] > 0).astype(int)
X_train.drop('PoolArea', axis=1, inplace=True)
X_valid.drop('PoolArea', axis=1, inplace=True)

# Общая оцнека дома (RForest -250 (16915), XGboost +96 (16119))
X_train['OverallScore'] = X_train['OverallQual'] * X_train['OverallCond']
X_valid['OverallScore'] = X_valid['OverallQual'] * X_valid['OverallCond']


In [199]:
num_columns = [col for col in X_train.columns if X_train[col].dtype in [int, float]]
all_cat_columns = [col for col in X_train.columns if X_train[col].dtype == 'str']

In [208]:
# missing = X_train.isna().sum()
# missing[all_cat_columns][missing > 0]|

In [202]:
# Работа с категориальными колонками
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('OHencoder', OneHotEncoder(handle_unknown='ignore'))
])



In [203]:
# Preprocessing & Pipeline
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_columns),
    ('cat_nom', cat_transformer, all_cat_columns)
])

my_model_random_forest = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('RForest', RandomForestRegressor(n_estimators=200,
        random_state=42))
])

my_model_XGBoost = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('XGboost', XGBRegressor(n_estimators = 200, max_depth = 5, learning_rate = 0.1,))
])

In [204]:
my_model_random_forest.fit(X_train, y_train)
predict = my_model_random_forest.predict(X_valid)
mea = mean_absolute_error(y_valid, predict)
mea

17053.61327054795

In [205]:
my_model_XGBoost.fit(X_train, y_train)
predict = my_model_XGBoost.predict(X_valid)
mea = mean_absolute_error(y_valid, predict)
mea

15780.359375

In [206]:
# mae = -1 * cross_val_score(my_model_random_forest, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
# mae.mean()

In [209]:
mae = -1 * cross_val_score(my_model_XGBoost, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
mae.mean()

np.float64(16782.444140625)